In [1]:
import hashlib

def sha1_hash(input_string: str) -> str:
    """
    Generates the SHA-1 hash of the input string and returns it as a hexadecimal string.

    Args:
    input_string (str): The string to be hashed.

    Returns:
    str: The hexadecimal representation of the SHA-1 hash of the input string.
    """
    # Create a SHA-1 hash object
    hash_object = hashlib.sha1()

    # Update the hash object with bytes of the input string. SHA-1
    # and other hash functions in hashlib work with bytes, not str.
    hash_object.update(input_string.encode('utf-8'))

    # Return the SHA-1 hash as a hexadecimal string
    return hash_object.hexdigest()

s = "a"
print(str(sha1_hash(s)))

# 6385036879

86f7e437faa5a7fce15d1ddcb9eaeaea377667b8


In [2]:
import math
import hashlib
from itertools import product

############################
# 1. DJB2 Hash in Python   #
############################

def sha1_hash(input_string: str) -> str:
    """
    Generates the SHA-1 hash of the input string and returns it as a hexadecimal string.

    Args:
    input_string (str): The string to be hashed.

    Returns:
    str: The hexadecimal representation of the SHA-1 hash of the input string.
    """
    # Create a SHA-1 hash object
    hash_object = hashlib.sha1()

    # Update the hash object with bytes of the input string. SHA-1
    # and other hash functions in hashlib work with bytes, not str.
    hash_object.update(input_string.encode('utf-8'))

    # Return the SHA-1 hash as a hexadecimal string
    return hash_object.hexdigest()

############################
# 2. Constructing Plaintext
############################

def make_plaintext(message: str) -> str:
    """
    Build p = (message, DJB2 hash of message).
    We separate them with a special delimiter, e.g. '|', so we can parse later.

    For instance, if message="hello", then
    p = "hello|<djb2-hash-value-of-hello>"
    """
    return message + str(sha1_hash(message))

############################
# 3. Transposition Cipher  #
############################

def build_key_map(key: str):
    sorted_key = sorted(list(key))
    return [sorted_key.index(k) for k in key]

def encrypt_transposition(plaintext: str, key: str) -> str:
    """Encrypt plaintext using a simple columnar transposition with 'key'."""
    n = len(key)
    key_map = build_key_map(key)
    extra = len(plaintext) % n
    if extra != 0:
        plaintext += 'x' * (n - extra)
    rows = [plaintext[i:i+n] for i in range(0, len(plaintext), n)]
    print(rows)
    ciphertext = []
    for col_index in range(n):
        actual_col = key_map.index(col_index)
        for row in rows:
            ciphertext.append(row[actual_col])
    return ''.join(ciphertext)

def decrypt_transposition(ciphertext: str, key: str) -> str:
    n = len(key)
    key_map = build_key_map(key)
    num_rows = len(ciphertext) // n

    cols = [''] * n
    idx = 0
    for col_index in range(n):
        actual_col = key_map.index(col_index)
        cols[actual_col] = ciphertext[idx : idx + num_rows]
        idx += num_rows

    plaintext_chars = []
    for r in range(num_rows):
        for c in range(n):
            plaintext_chars.append(cols[c][r])
    return ''.join(plaintext_chars)

############################
# 4. Verifying the Property
############################

def remove_x(decrypted_text):
  c=0
  for i in decrypted_text[::-1]:
    c+=1
    if i!='x':
      break
  midtext = decrypted_text[:len(decrypted_text)-c+1]
  return midtext

def passes_property_pi(decrypted_text: str) -> bool:

  original_text = decrypted_text[:-40]
  hash_value = decrypted_text[-40:]

  return (hash_value == str(sha1_hash(original_text)))


    # """
    # Our property pi: the decrypted text must split into
    #    (original_message)|(hash_of_original_message)
    # and match exactly the DJB2 hash of 'original_message'.
    # """
    # if '|' not in decrypted_text:
    #     return False

    # # Split on the rightmost '|' in case the message accidentally has '|'
    # # This ensures we get the final chunk as the stored hash
    # main_part, hash_part_str = decrypted_text.rsplit('|', 1)

    # # If hash_part_str is not purely digits, it might fail int() conversion
    # if not hash_part_str.isdigit():
    #     return False

    # # Recompute the hash of main_part
    # computed_hash = sha1_hash(main_part)
    # # Compare to stored value
    # return (computed_hash == int(hash_part_str))


############################
# 5. Simple Testing        #
############################

def test_encryption_decryption():
    """
    Quick test of encryption & decryption with a known key.
    """
    key = "bdac"
    messages = [
        "hello",
        "testmessage",
        "abcxyz",
        "transpositioncipher",
        "bruteforcefun"
    ]

    for i, msg in enumerate(messages, 1):
        print(f"\n--- Message {i} ---")
        # Build p = message|<hash>
        print("message: ", msg)
        p = make_plaintext(msg)
        print(f"Plaintext : {p}")

        c = encrypt_transposition(p, key)
        print(f"Ciphertext: {c}")

        p_decrypted = decrypt_transposition(c, key)
        print(f"Decrypted : {p_decrypted}") #hello210714636441xxx
        decrypt_nox = remove_x(p_decrypted)
        print(f"Plain Text : {decrypt_nox}")
        original_text = decrypt_nox[:-40]
        print(f"Original Text : {original_text}")
        print(f"Matches original plaintext? {p == decrypt_nox}")

        # Also check the property:


        print(f"Passes property pi? {passes_property_pi(decrypt_nox)}")





In [3]:
print("=== Testing encryption/decryption with DJB2 hash ===")
test_encryption_decryption()

print("\n=== Demo brute-force attack (small key space) ===")


=== Testing encryption/decryption with DJB2 hash ===

--- Message 1 ---
message:  hello
Plaintext : helloaaf4c61ddcc5e8a2dabede0f3b482cd9aea9434d
['hell', 'oaaf', '4c61', 'ddcc', '5e8a', '2dab', 'ede0', 'f3b4', '82cd', '9aea', '9434', 'dxxx']
Ciphertext: la6c8aebce3xho4d52ef899dlf1cab04da4xeacdedd32a4x
Decrypted : helloaaf4c61ddcc5e8a2dabede0f3b482cd9aea9434dxxx
Plain Text : helloaaf4c61ddcc5e8a2dabede0f3b482cd9aea9434d
Original Text : hello
Matches original plaintext? True
Passes property pi? True

--- Message 2 ---
message:  testmessage
Plaintext : testmessaged9d15be1d634e5cc656e640e5501a7bd925ab51e
['test', 'mess', 'aged', '9d15', 'be1d', '634e', '5cc6', '56e6', '40e5', '501a', '7bd9', '25ab', '51ex']
Ciphertext: sse114cee1daetma9b65545725tsd5de665a9bxeegde3c600b51
Decrypted : testmessaged9d15be1d634e5cc656e640e5501a7bd925ab51ex
Plain Text : testmessaged9d15be1d634e5cc656e640e5501a7bd925ab51e
Original Text : testmessage
Matches original plaintext? True
Passes property pi? True

--- 

In [4]:
from itertools import permutations

############################
# 6. Brute Force Attack    #
############################

def brute_force_attack(ciphertext, alphabet='abcd', max_key_length=4):
    """
    Brute force to find the key that decrypts 'ciphertext' correctly according to the property pi.

    Args:
    ciphertext (str): The encrypted text.
    alphabet (str): A string of characters used to form keys.
    max_key_length (int): The maximum length of keys to try.

    Returns:
    str or None: The correct key if found, otherwise None.
    """
    # Generate all permutations of all lengths up to max_key_length from the alphabet
    for length in range(1, max_key_length + 1):
        for key_tuple in permutations(alphabet, length):
            key = ''.join(key_tuple)
            # Decrypt the ciphertext with the generated key
            decrypted_text = decrypt_transposition(ciphertext, key)
            decrypted_text = remove_x(decrypted_text)
            # Check if the decrypted text passes the property pi
            if passes_property_pi(decrypted_text):
                return key
    return None  # No valid key found

############################
# 7. Demonstrating Brute Force Attack
############################

def demo_brute_force():
    """
    Demonstration of brute forcing to find a key for a given ciphertext.
    """
    key = "bdac"
    message = "testmessage"
    plaintext = make_plaintext(message)
    ciphertext = encrypt_transposition(plaintext, key)
    print(f"Ciphertext: {ciphertext}")

    # Attempt to discover the key
    discovered_key = brute_force_attack(ciphertext)
    if discovered_key:
        print(f"Key found: {discovered_key}")
        decrypted_text = decrypt_transposition(ciphertext, discovered_key)
        decrypted_text = remove_x(decrypted_text)
        print(f"Decrypted Text: {decrypted_text}")
    else:
        print("No valid key found.")

if __name__ == "__main__":
    test_encryption_decryption()
    demo_brute_force()



--- Message 1 ---
message:  hello
Plaintext : helloaaf4c61ddcc5e8a2dabede0f3b482cd9aea9434d
['hell', 'oaaf', '4c61', 'ddcc', '5e8a', '2dab', 'ede0', 'f3b4', '82cd', '9aea', '9434', 'dxxx']
Ciphertext: la6c8aebce3xho4d52ef899dlf1cab04da4xeacdedd32a4x
Decrypted : helloaaf4c61ddcc5e8a2dabede0f3b482cd9aea9434dxxx
Plain Text : helloaaf4c61ddcc5e8a2dabede0f3b482cd9aea9434d
Original Text : hello
Matches original plaintext? True
Passes property pi? True

--- Message 2 ---
message:  testmessage
Plaintext : testmessaged9d15be1d634e5cc656e640e5501a7bd925ab51e
['test', 'mess', 'aged', '9d15', 'be1d', '634e', '5cc6', '56e6', '40e5', '501a', '7bd9', '25ab', '51ex']
Ciphertext: sse114cee1daetma9b65545725tsd5de665a9bxeegde3c600b51
Decrypted : testmessaged9d15be1d634e5cc656e640e5501a7bd925ab51ex
Plain Text : testmessaged9d15be1d634e5cc656e640e5501a7bd925ab51e
Original Text : testmessage
Matches original plaintext? True
Passes property pi? True

--- Message 3 ---
message:  abcxyz
Plaintext : abcxyz0e3b

In [5]:
# ############################
# # 6. Brute Force Attack    #
# ############################

# def brute_force_attack(ciphertexts):
#     """
#     Brute force keys in a small subset: {a,b,c,d}^length up to 2,
#     for demonstration. In a real scenario (length <= 9, alphabet=a..z),
#     you'd need a more optimized search.

#     For each candidate key:
#       - Decrypt each ciphertext c_i with that key
#       - Check if passes_property_pi
#       - If ALL 5 ciphertexts pass pi, we declare success
#     """
#     alphabet_subset = ['a','b','c','d']
#     max_key_length = 4

#     for length in range(1, max_key_length + 1):
#         # Generate all possible keys of 'length' from our small subset
#         for key_tuple in product(alphabet_subset, repeat=length):
#             candidate_key = ''.join(key_tuple)

#             all_good = True
#             for ctext in ciphertexts:
#                 p_candidate = decrypt_transposition(ctext, candidate_key)
#                 if not passes_property_pi(p_candidate):
#                     all_good = False
#                     break

#             if all_good:
#                 return candidate_key

#     return None  # No valid key found in the tested space

# def demo_brute_force():
#     """
#     Demonstrate brute-forcing a small key space:
#       - We create 5 ciphertexts with a real key 'real_key'.
#       - We try to discover that key from scratch.
#     """
#     real_key = "bdac"
#     messages = [
#         "hello",
#         "testmessage",
#         "abcxyz",
#         "transpositioncipher",
#         "bruteforcefun"
#     ]

#     # Build 5 plaintexts p = message|<sha1_hash(message)>
#     plaintexts = [make_plaintext(m) for m in messages]
#     # Encrypt them all
#     ciphertexts = [encrypt_transposition(p, real_key) for p in plaintexts]

#     # Try to rediscover the key by brute force
#     discovered_key = brute_force_attack(ciphertexts)
#     if discovered_key is not None:
#         print(f"[*] Found candidate key: {discovered_key}")
#         # Verify by decrypting all ciphertexts
#         for i, c in enumerate(ciphertexts, 1):
#             dec_p = decrypt_transposition(c, discovered_key)
#             print(f"Ciphertext {i} => Plaintext: {dec_p}")
#     else:
#         print("[!] No valid key found in the tested space.")

In [6]:
# demo_brute_force()